# Understanding async and await in Python

This notebook demonstrates how `async` and `await` work in Python, and what it means for code to be "non-blocking" or "unblocking the rest of the program".

In [ ]:
import time
import asyncio

## Synchronous (blocking) example

In synchronous code, if you call a function that takes a long time (like `time.sleep`), your whole program waits ("blocks") until that function finishes.

In [ ]:
def sync_task(name, seconds):
    print(f"Starting {name}")
    time.sleep(seconds)
    print(f"Finished {name}")

# Run two tasks synchronously
sync_task("Task 1", 2)
sync_task("Task 2", 2)

Starting Task 1
Finished Task 1
Starting Task 2
Finished Task 2


## Asynchronous (non-blocking) example

With `async` and `await`, you can let your program do other things while waiting for a task to finish. This is useful for I/O-bound tasks (like waiting for a network response).

In [ ]:
async def async_task(name, seconds):
    print(f"Starting {name}")
    await asyncio.sleep(seconds)
    print(f"Finished {name}")

# Run two tasks asynchronously
async def main():
    await asyncio.gather(
        async_task("Task 1", 2),
        async_task("Task 2", 2)
    )

await main()

Starting Task 1
Starting Task 2
Finished Task 1
Finished Task 2


## What does "unblocking" mean?

In the async example, while one task is waiting ("sleeping"), the program can start or continue other tasks. This is called "non-blocking" or "unblocking the rest of the program". In contrast, synchronous code makes everything wait for each task to finish before starting the next.

You can see that both tasks start together and finish together in the async example, but run one after the other in the sync example.

## Try changing the sleep times and see how the behavior changes!

## Async Iteration: `async for`

Just like you can use `for` to loop over items in a list, you can use `async for` to loop over items that arrive asynchronously (for example, from a network stream or an async generator).

This is useful when you want to process data as it becomes available, without blocking your program while waiting for each item.

In [ ]:
async def async_generator():
    for i in range(3):
        await asyncio.sleep(10)  # Simulate waiting for data
        yield i

async def main_async_for():
    print("Starting async for loop...")
    async for value in async_generator():
        print(f"Received value: {value}")
    print("Async for loop done!")

await main_async_for()

In [9]:
print("This will run immediately after starting the async for loop.")

This will run immediately after starting the async for loop.


In the example above:
- `async_generator` yields a value every second.
- The `async for` loop processes each value as soon as it is available, without blocking the rest of the program.
- This is similar to how you might process streamed data from a server or API.

**Key point:** Each iteration waits for the next value, but the rest of your program can remain responsive or do other work while waiting.

## What happens if you don't use `await`?

If you call an async function without `await`, it returns a coroutine object immediately and does NOT start running. The rest of your program (including the next cell in a notebook) can run, but the async function won't actually execute until you `await` it or schedule it with an event loop.

In [25]:
async def async_task(name, seconds):
    print(f"Starting {name}")
    await asyncio.sleep(seconds)
    print(f"Finished {name}")

# Call async_task without await
coroutine = async_task("Task 1", 2)
print("This prints immediately, async_task hasn't started yet!")

This prints immediately, async_task hasn't started yet!


In [26]:
# Now actually run the coroutine
await coroutine

Starting Task 1
Finished Task 1


**Note:**
- When you call an async function without `await`, it does NOT run. It just creates a coroutine object.
- The rest of your code (including the next cell) can run while the coroutine is waiting to be scheduled.
- In a notebook, if you don't `await` the coroutine, it won't execute until you do.

**Python GIL:**
- The Global Interpreter Lock (GIL) does not prevent async code from running concurrently. Async code is about cooperative multitasking, not parallel threads.
- Async tasks share the same thread and take turns running, so you can have multiple tasks "in progress" as long as you use `await` or schedule them with the event loop.

Try running the code above and see how the print statements behave!